<a href="https://colab.research.google.com/github/Likith-Reddy25/Summer-Intern/blob/main/codes/MNIST_Covariant_QKE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MNIST Covariant Feature Map QKE

In [ ]:
"""
CovariantFeatureMap plain-QKE reproduction of the MNIST-PCA-8 row in
Alvarez-Estevez (2025), Table 2. Fully self-contained: downloads MNIST,
builds the dataset, builds the quantum kernel, runs n repetitions, prints
the summary table. Just run this one file top to bottom (works in
Colab/Jupyter or as a plain script).
"""

# ------------------------------------------------------------------
# 0. Install deps (safe to re-run; skips if already installed)
# ------------------------------------------------------------------
import subprocess, sys
def _pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import qiskit, qiskit_machine_learning  # noqa
except ImportError:
    _pip_install(["qiskit==1.1.0", "qiskit-machine-learning==0.7.2", "qiskit-algorithms==0.3.0"])

# ------------------------------------------------------------------
# 1. Download MNIST (idempotent, absolute path so it's found later)
# ------------------------------------------------------------------
import os, urllib.request

MNIST_PKL = os.path.join(os.getcwd(), "mnist.pkl.gz")
MNIST_URL = ("https://raw.githubusercontent.com/mnielsen/"
             "neural-networks-and-deep-learning/master/data/mnist.pkl.gz")

if not os.path.exists(MNIST_PKL) or os.path.getsize(MNIST_PKL) < 10_000_000:
    print(f"Downloading MNIST to {MNIST_PKL} ...")
    urllib.request.urlretrieve(MNIST_URL, MNIST_PKL)
print("MNIST file ready:", MNIST_PKL, os.path.getsize(MNIST_PKL), "bytes")

# ------------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------------
import gzip
import pickle
import time
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_machine_learning.kernels import FidelityStatevectorKernel

from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

# ------------------------------------------------------------------
# 3. CovariantFeatureMap circuit + quantum kernel  (eqs. 16-19)
# ------------------------------------------------------------------
def num_qubits_for(num_features: int) -> int:
    if num_features % 2 != 0:
        raise ValueError("CovariantFeatureMap requires an even number of features.")
    return num_features // 2


def build_covariant_circuit(num_features: int, ansatz: str = None) -> QuantumCircuit:
    """U_theta(x) = D(x) * V(theta).  ansatz=None -> plain QKE (theta fixed at 0)."""
    n = num_qubits_for(num_features)
    x = ParameterVector("x", num_features)
    qc = QuantumCircuit(n, name="CovariantFeatureMap")

    # D(x): eq. (16)
    for k in range(n):
        qc.rx(x[2 * k], k)
        qc.rz(x[2 * k + 1], k)

    # linear entangling layer, eq. (18)
    for k in range(n - 1):
        qc.cz(k, k + 1)

    # V(theta): fiducial state, eq. (20)/(21)
    if ansatz in ("shared", "dedicated"):
        if ansatz == "shared":
            theta = ParameterVector("theta", 3)
            for k in range(n):
                qc.u(theta[0], theta[1], theta[2], k)
        else:
            theta = ParameterVector("theta", 3 * n)
            for k in range(n):
                qc.u(theta[3 * k], theta[3 * k + 1], theta[3 * k + 2], k)
        qc.metadata = {"feature_params": list(x), "theta_params": list(theta)}
    else:
        qc.metadata = {"feature_params": list(x), "theta_params": []}
    return qc


def make_kernel(num_features: int, ansatz: str = None, theta_values=None):
    """Plain QKE (ansatz=None) with CovariantFeatureMap, theta = 0 -> identity fiducial state."""
    qc = build_covariant_circuit(num_features, ansatz=ansatz)
    feat_params = qc.metadata["feature_params"]
    theta_params = qc.metadata["theta_params"]
    if theta_params:
        if theta_values is None:
            theta_values = np.zeros(len(theta_params))
        qc = qc.assign_parameters(dict(zip(theta_params, theta_values)))
    return FidelityStatevectorKernel(feature_map=qc)


# ------------------------------------------------------------------
# 4. Dataset: MNIST 3-vs-5, PCA-8  (Table 1 / Section III-B2)
# ------------------------------------------------------------------
N_COMPONENTS = 8            # -> MNIST-PCA-8 (4 qubits)
N_TRAIN, N_TEST = 250, 250
C_GRID = [0.01, 0.1, 1, 10, 100]
LAMBDA_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]
N_FOLDS = 5


def load_mnist_3_vs_5(path=MNIST_PKL):
    with gzip.open(path, "rb") as f:
        train, val, test = pickle.load(f, encoding="latin1")
    X = np.concatenate([train[0], val[0], test[0]], axis=0)
    y = np.concatenate([train[1], val[1], test[1]], axis=0)
    mask = (y == 3) | (y == 5)
    X, y = X[mask], y[mask]
    y = np.where(y == 3, -1, 1)     # eq. (1)-(4) labels in {-1, +1}
    return X, y


def build_pca8_pool(X, y, pca_fit_size=6000, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X), size=min(pca_fit_size, len(X)), replace=False)
    pca = PCA(n_components=N_COMPONENTS, random_state=seed)
    pca.fit(X[idx])
    return pca.transform(X), y, pca


# ------------------------------------------------------------------
# 5. QKE evaluation, CV hyperparameter search, one repetition
# ------------------------------------------------------------------
def quantum_kernel_matrix(kernel, X_a, X_b, lam):
    return kernel.evaluate(x_vec=lam * X_a, y_vec=lam * X_b)


def cv_select_hyperparams(kernel, X_tr, y_tr, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    folds = list(skf.split(X_tr, y_tr))
    best_acc, best_lam, best_C = -1, LAMBDA_GRID[0], C_GRID[0]
    for lam in LAMBDA_GRID:
        K_full = quantum_kernel_matrix(kernel, X_tr, X_tr, lam)
        for C in C_GRID:
            accs = []
            for tr_idx, val_idx in folds:
                K_tr = K_full[np.ix_(tr_idx, tr_idx)]
                K_val = K_full[np.ix_(val_idx, tr_idx)]
                svc = SVC(kernel="precomputed", C=C)
                svc.fit(K_tr, y_tr[tr_idx])
                accs.append(accuracy_score(y_tr[val_idx], svc.predict(K_val)))
            mean_acc = float(np.mean(accs))
            if mean_acc > best_acc:
                best_acc, best_lam, best_C = mean_acc, lam, C
    return best_lam, best_C


def run_repetition(kernel, X_pool, y_pool, seed):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_pool, y_pool, train_size=N_TRAIN, test_size=N_TEST,
        stratify=y_pool, random_state=seed,
    )
    scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    lam, C = cv_select_hyperparams(kernel, X_tr, y_tr, seed)

    K_train = quantum_kernel_matrix(kernel, X_tr, X_tr, lam)
    K_test = quantum_kernel_matrix(kernel, X_te, X_tr, lam)

    svc = SVC(kernel="precomputed", C=C)
    svc.fit(K_train, y_tr)
    pred_tr = svc.predict(K_train)
    pred_te = svc.predict(K_test)

    return {
        "acc_train": accuracy_score(y_tr, pred_tr),
        "acc_test": accuracy_score(y_te, pred_te),
        "kappa_train": cohen_kappa_score(y_tr, pred_tr),
        "kappa_test": cohen_kappa_score(y_te, pred_te),
        "f1_train": f1_score(y_tr, pred_tr, average="macro"),
        "f1_test": f1_score(y_te, pred_te, average="macro"),
        "lambda": lam, "C": C,
    }


# ------------------------------------------------------------------
# 6. Run everything
# ------------------------------------------------------------------
REPS = 10  # <-- change this if you want fewer/more repetitions
SEED = 0

print("\nLoading MNIST 3-vs-5 and building PCA-8 pool ...")
X, y = load_mnist_3_vs_5()
X_pca, y_pca, pca = build_pca8_pool(X, y, seed=SEED)
print(f"Pool size: {len(X_pca)}  (explained var. ratio sum = "
      f"{pca.explained_variance_ratio_.sum():.3f})")

kernel = make_kernel(num_features=N_COMPONENTS, ansatz=None)  # plain QKE

all_metrics = []
t0 = time.time()
for r in range(REPS):
    m = run_repetition(kernel, X_pca, y_pca, seed=SEED + r)
    all_metrics.append(m)
    print(f"  rep {r+1:2d}/{REPS}  acc_test={m['acc_test']:.3f}  "
          f"lambda={m['lambda']}  C={m['C']}  ({time.time()-t0:.1f}s elapsed)")

keys = ["acc_train", "acc_test", "kappa_train", "kappa_test", "f1_train", "f1_test"]
summary = {k: (np.mean([m[k] for m in all_metrics]),
                np.std([m[k] for m in all_metrics])) for k in keys}

print("\n=== Table 2 format ===")
print(f"{'Dataset':<14}{'Feature map':<20}{'Strategy':<10}"
      f"{'acc_TR':<16}{'acc_TS':<16}{'kappa_TR':<16}{'kappa_TS':<16}"
      f"{'F1_TR':<16}{'F1_TS':<16}")
print(f"{'MNIST-PCA-8':<14}{'CovariantFeatureMap':<20}{'-':<10}"
      f"{summary['acc_train'][0]:.3f}({summary['acc_train'][1]:.3f})   "
      f"{summary['acc_test'][0]:.3f}({summary['acc_test'][1]:.3f})   "
      f"{summary['kappa_train'][0]:.3f}({summary['kappa_train'][1]:.3f})   "
      f"{summary['kappa_test'][0]:.3f}({summary['kappa_test'][1]:.3f})   "
      f"{summary['f1_train'][0]:.3f}({summary['f1_train'][1]:.3f})   "
      f"{summary['f1_test'][0]:.3f}({summary['f1_test'][1]:.3f})")

MNIST file ready: /content/mnist.pkl.gz 17051982 bytes

Loading MNIST 3-vs-5 and building PCA-8 pool ...
Pool size: 13454  (explained var. ratio sum = 0.476)
  rep  1/10  acc_test=0.944  lambda=0.5  C=1  (9.8s elapsed)
  rep  2/10  acc_test=0.964  lambda=0.5  C=10  (15.8s elapsed)
  rep  3/10  acc_test=0.940  lambda=0.5  C=1  (21.9s elapsed)
  rep  4/10  acc_test=0.940  lambda=0.5  C=1  (27.2s elapsed)
  rep  5/10  acc_test=0.880  lambda=0.5  C=10  (33.7s elapsed)
  rep  6/10  acc_test=0.928  lambda=0.5  C=10  (39.1s elapsed)
  rep  7/10  acc_test=0.964  lambda=0.5  C=1  (45.7s elapsed)
  rep  8/10  acc_test=0.936  lambda=0.5  C=1  (51.2s elapsed)
  rep  9/10  acc_test=0.936  lambda=0.5  C=10  (57.7s elapsed)
  rep 10/10  acc_test=0.928  lambda=0.5  C=1  (63.1s elapsed)

=== Table 2 format ===
Dataset       Feature map         Strategy  acc_TR          acc_TS          kappa_TR        kappa_TS        F1_TR           F1_TS           
MNIST-PCA-8   CovariantFeatureMap -         0.968(0.01

MNIST Covariant Feature Map Shared 3


In [ ]:
"""
CovariantFeatureMap plain-QKE reproduction of the MNIST-PCA-8 row in
Alvarez-Estevez (2025), Table 2. Fully self-contained: downloads MNIST,
builds the dataset, builds the quantum kernel, runs n repetitions, prints
the summary table. Just run this one file top to bottom (works in
Colab/Jupyter or as a plain script).
"""

# ------------------------------------------------------------------
# 0. Install deps (safe to re-run; skips if already installed)
# ------------------------------------------------------------------
import subprocess, sys
def _pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import qiskit, qiskit_machine_learning  # noqa
except ImportError:
    _pip_install(["qiskit==1.1.0", "qiskit-machine-learning==0.7.2", "qiskit-algorithms==0.3.0"])

# ------------------------------------------------------------------
# 1. Download MNIST (idempotent, absolute path so it's found later)
# ------------------------------------------------------------------
import os, urllib.request

MNIST_PKL = os.path.join(os.getcwd(), "mnist.pkl.gz")
MNIST_URL = ("https://raw.githubusercontent.com/mnielsen/"
             "neural-networks-and-deep-learning/master/data/mnist.pkl.gz")

if not os.path.exists(MNIST_PKL) or os.path.getsize(MNIST_PKL) < 10_000_000:
    print(f"Downloading MNIST to {MNIST_PKL} ...")
    urllib.request.urlretrieve(MNIST_URL, MNIST_PKL)
print("MNIST file ready:", MNIST_PKL, os.path.getsize(MNIST_PKL), "bytes")

# ------------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------------
import gzip
import pickle
import time
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_machine_learning.kernels import FidelityStatevectorKernel

from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

# ------------------------------------------------------------------
# 3. CovariantFeatureMap circuit + quantum kernel  (eqs. 16-19)
# ------------------------------------------------------------------
def num_qubits_for(num_features: int) -> int:
    if num_features % 2 != 0:
        raise ValueError("CovariantFeatureMap requires an even number of features.")
    return num_features // 2


def build_covariant_circuit(num_features: int, ansatz: str = None) -> QuantumCircuit:
    """U_theta(x) = D(x) * V(theta).  ansatz=None -> plain QKE (theta fixed at 0)."""
    n = num_qubits_for(num_features)
    x = ParameterVector("x", num_features)
    qc = QuantumCircuit(n, name="CovariantFeatureMap")

    # D(x): eq. (16)
    for k in range(n):
        qc.rx(x[2 * k], k)
        qc.rz(x[2 * k + 1], k)

    # linear entangling layer, eq. (18)
    for k in range(n - 1):
        qc.cz(k, k + 1)

    # V(theta): fiducial state, eq. (20)/(21)
    if ansatz in ("shared", "dedicated"):
        if ansatz == "shared":
            theta = ParameterVector("theta", 3)
            for k in range(n):
                qc.u(theta[0], theta[1], theta[2], k)
        else:
            theta = ParameterVector("theta", 3 * n)
            for k in range(n):
                qc.u(theta[3 * k], theta[3 * k + 1], theta[3 * k + 2], k)
        qc.metadata = {"feature_params": list(x), "theta_params": list(theta)}
    else:
        qc.metadata = {"feature_params": list(x), "theta_params": []}
    return qc


def make_kernel(num_features: int, ansatz: str = None, theta_values=None):
    """Plain QKE (ansatz=None) with CovariantFeatureMap, theta = 0 -> identity fiducial state."""
    qc = build_covariant_circuit(num_features, ansatz=ansatz)
    feat_params = qc.metadata["feature_params"]
    theta_params = qc.metadata["theta_params"]
    if theta_params:
        if theta_values is None:
            theta_values = np.zeros(len(theta_params))
        qc = qc.assign_parameters(dict(zip(theta_params, theta_values)))
    return FidelityStatevectorKernel(feature_map=qc)


# ------------------------------------------------------------------
# 4. Dataset: MNIST 3-vs-5, PCA-8  (Table 1 / Section III-B2)
# ------------------------------------------------------------------
N_COMPONENTS = 8            # -> MNIST-PCA-8 (4 qubits)
N_TRAIN, N_TEST = 250, 250
C_GRID = [0.01, 0.1, 1, 10, 100]
LAMBDA_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]
N_FOLDS = 5


def load_mnist_3_vs_5(path=MNIST_PKL):
    with gzip.open(path, "rb") as f:
        train, val, test = pickle.load(f, encoding="latin1")
    X = np.concatenate([train[0], val[0], test[0]], axis=0)
    y = np.concatenate([train[1], val[1], test[1]], axis=0)
    mask = (y == 3) | (y == 5)
    X, y = X[mask], y[mask]
    y = np.where(y == 3, -1, 1)     # eq. (1)-(4) labels in {-1, +1}
    return X, y


def build_pca8_pool(X, y, pca_fit_size=6000, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X), size=min(pca_fit_size, len(X)), replace=False)
    pca = PCA(n_components=N_COMPONENTS, random_state=seed)
    pca.fit(X[idx])
    return pca.transform(X), y, pca


# ------------------------------------------------------------------
# 5. QKE evaluation, CV hyperparameter search, one repetition
# ------------------------------------------------------------------
def quantum_kernel_matrix(kernel, X_a, X_b, lam):
    return kernel.evaluate(x_vec=lam * X_a, y_vec=lam * X_b)


def cv_select_hyperparams(kernel, X_tr, y_tr, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    folds = list(skf.split(X_tr, y_tr))
    best_acc, best_lam, best_C = -1, LAMBDA_GRID[0], C_GRID[0]
    for lam in LAMBDA_GRID:
        K_full = quantum_kernel_matrix(kernel, X_tr, X_tr, lam)
        for C in C_GRID:
            accs = []
            for tr_idx, val_idx in folds:
                K_tr = K_full[np.ix_(tr_idx, tr_idx)]
                K_val = K_full[np.ix_(val_idx, tr_idx)]
                svc = SVC(kernel="precomputed", C=C)
                svc.fit(K_tr, y_tr[tr_idx])
                accs.append(accuracy_score(y_tr[val_idx], svc.predict(K_val)))
            mean_acc = float(np.mean(accs))
            if mean_acc > best_acc:
                best_acc, best_lam, best_C = mean_acc, lam, C
    return best_lam, best_C


def run_repetition(kernel, X_pool, y_pool, seed):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_pool, y_pool, train_size=N_TRAIN, test_size=N_TEST,
        stratify=y_pool, random_state=seed,
    )
    scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    lam, C = cv_select_hyperparams(kernel, X_tr, y_tr, seed)

    K_train = quantum_kernel_matrix(kernel, X_tr, X_tr, lam)
    K_test = quantum_kernel_matrix(kernel, X_te, X_tr, lam)

    svc = SVC(kernel="precomputed", C=C)
    svc.fit(K_train, y_tr)
    pred_tr = svc.predict(K_train)
    pred_te = svc.predict(K_test)

    return {
        "acc_train": accuracy_score(y_tr, pred_tr),
        "acc_test": accuracy_score(y_te, pred_te),
        "kappa_train": cohen_kappa_score(y_tr, pred_tr),
        "kappa_test": cohen_kappa_score(y_te, pred_te),
        "f1_train": f1_score(y_tr, pred_tr, average="macro"),
        "f1_test": f1_score(y_te, pred_te, average="macro"),
        "lambda": lam, "C": C,
    }


# ------------------------------------------------------------------
# 6. Run everything
# ------------------------------------------------------------------
REPS = 30   # <-- change this if you want fewer/more repetitions
SEED = 0

print("\nLoading MNIST 3-vs-5 and building PCA-8 pool ...")
X, y = load_mnist_3_vs_5()
X_pca, y_pca, pca = build_pca8_pool(X, y, seed=SEED)
print(f"Pool size: {len(X_pca)}  (explained var. ratio sum = "
      f"{pca.explained_variance_ratio_.sum():.3f})")

kernel = make_kernel(num_features=N_COMPONENTS, ansatz=None)  # plain QKE

all_metrics = []
t0 = time.time()
for r in range(REPS):
    m = run_repetition(kernel, X_pca, y_pca, seed=SEED + r)
    all_metrics.append(m)
    print(f"  rep {r+1:2d}/{REPS}  acc_test={m['acc_test']:.3f}  "
          f"lambda={m['lambda']}  C={m['C']}  ({time.time()-t0:.1f}s elapsed)")

keys = ["acc_train", "acc_test", "kappa_train", "kappa_test", "f1_train", "f1_test"]
summary = {k: (np.mean([m[k] for m in all_metrics]),
                np.std([m[k] for m in all_metrics])) for k in keys}

print("\n=== Table 2 format ===")
print(f"{'Dataset':<14}{'Feature map':<20}{'Strategy':<10}"
      f"{'acc_TR':<16}{'acc_TS':<16}{'kappa_TR':<16}{'kappa_TS':<16}"
      f"{'F1_TR':<16}{'F1_TS':<16}")
print(f"{'MNIST-PCA-8':<14}{'CovariantFeatureMap':<20}{'-':<10}"
      f"{summary['acc_train'][0]:.3f}({summary['acc_train'][1]:.3f})   "
      f"{summary['acc_test'][0]:.3f}({summary['acc_test'][1]:.3f})   "
      f"{summary['kappa_train'][0]:.3f}({summary['kappa_train'][1]:.3f})   "
      f"{summary['kappa_test'][0]:.3f}({summary['kappa_test'][1]:.3f})   "
      f"{summary['f1_train'][0]:.3f}({summary['f1_train'][1]:.3f})   "
      f"{summary['f1_test'][0]:.3f}({summary['f1_test'][1]:.3f})")

MNIST file ready: /content/mnist.pkl.gz 17051982 bytes

Loading MNIST 3-vs-5 and building PCA-8 pool ...
Pool size: 13454  (explained var. ratio sum = 0.476)
  rep  1/30  acc_test=0.944  lambda=0.5  C=1  (6.0s elapsed)
  rep  2/30  acc_test=0.964  lambda=0.5  C=10  (13.0s elapsed)
  rep  3/30  acc_test=0.940  lambda=0.5  C=1  (18.7s elapsed)
  rep  4/30  acc_test=0.940  lambda=0.5  C=1  (25.7s elapsed)
  rep  5/30  acc_test=0.880  lambda=0.5  C=10  (31.4s elapsed)
  rep  6/30  acc_test=0.928  lambda=0.5  C=10  (38.3s elapsed)
  rep  7/30  acc_test=0.964  lambda=0.5  C=1  (43.9s elapsed)
  rep  8/30  acc_test=0.936  lambda=0.5  C=1  (50.3s elapsed)
  rep  9/30  acc_test=0.936  lambda=0.5  C=10  (56.6s elapsed)
  rep 10/30  acc_test=0.928  lambda=0.5  C=1  (63.2s elapsed)
  rep 11/30  acc_test=0.940  lambda=0.5  C=0.1  (69.3s elapsed)
  rep 12/30  acc_test=0.924  lambda=0.5  C=100  (75.6s elapsed)
  rep 13/30  acc_test=0.940  lambda=0.5  C=1  (81.8s elapsed)
  rep 14/30  acc_test=0.932  

MNIST PCA 8 Dedicated 12


In [ ]:
"""
QKT (weighted-alignment Quantum Kernel Training) reproduction of the
"MNIST-PCA-8 / CovariantFeatureMap / qSVM_COV_opt_d (dedicated, 12 params)"
row of Table 2 in Alvarez-Estevez (2025), "Benchmarking Quantum Machine
Learning Kernel Training for Classification Tasks".

Fully self-contained: installs deps, downloads MNIST, builds the
CovariantFeatureMap circuit with the "dedicated" fiducial-state ansatz
(eq. 21: each of the n=4 qubits (MNIST-PCA-8 -> 4 qubits) gets its own
independent Euler-angle triple -> 3*4 = 12 trainable parameters total),
trains theta with 2-SPSA + weighted kernel-target alignment (SVCLoss =
eq. 13), then runs the usual (C, lambda) grid search + n repetitions +
Table-2-style summary.

IMPORTANT - RUNTIME:
QKT is expensive: each SPSA iteration needs ~2-4 full training-kernel
evaluations. At 250 training points, one iteration takes roughly 4-7s
on CPU statevector simulation. The paper uses maxiter=400 and reps=30,
which would take many hours here. Defaults below (QKT_MAXITER, REPS)
are set small so you can sanity-check the pipeline quickly; bump them
up (see bottom of file) to approach paper-scale numbers, ideally on a
machine you can leave running.
    QKT_MAXITER=400, REPS=30  ~ matches the paper's protocol, but is a
    multi-hour run per repetition count -> plan accordingly.
"""

# ------------------------------------------------------------------
# 0. Install deps (idempotent)
# ------------------------------------------------------------------
import subprocess, sys
def _pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import qiskit, qiskit_machine_learning, qiskit_algorithms  # noqa
except ImportError:
    _pip_install(["qiskit==1.1.0", "qiskit-machine-learning==0.7.2", "qiskit-algorithms==0.3.0"])

# ------------------------------------------------------------------
# 1. Download MNIST (idempotent, absolute path)
# ------------------------------------------------------------------
import os, urllib.request

MNIST_PKL = os.path.join(os.getcwd(), "mnist.pkl.gz")
MNIST_URL = ("https://raw.githubusercontent.com/mnielsen/"
             "neural-networks-and-deep-learning/master/data/mnist.pkl.gz")

if not os.path.exists(MNIST_PKL) or os.path.getsize(MNIST_PKL) < 10_000_000:
    print(f"Downloading MNIST to {MNIST_PKL} ...")
    urllib.request.urlretrieve(MNIST_URL, MNIST_PKL)
print("MNIST file ready:", MNIST_PKL, os.path.getsize(MNIST_PKL), "bytes")

# ------------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------------
import gzip
import pickle
import time
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_machine_learning.kernels import (
    FidelityStatevectorKernel,
    TrainableFidelityStatevectorKernel,
)
from qiskit_machine_learning.kernels.algorithms import QuantumKernelTrainer
from qiskit_algorithms.optimizers import SPSA

from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

# ------------------------------------------------------------------
# 3. CovariantFeatureMap circuit  (eqs. 16, 18, 20)
# ------------------------------------------------------------------
def num_qubits_for(num_features: int) -> int:
    if num_features % 2 != 0:
        raise ValueError("CovariantFeatureMap requires an even number of features.")
    return num_features // 2


def build_covariant_circuit_dedicated(num_features: int) -> QuantumCircuit:
    """
    U_theta(x) = D(x) * Vdedicated(theta), eq. (16),(18),(21).
    Vdedicated: each of the n qubits gets its own independent
    (theta1,theta2,theta3) Euler-angle rotation -> 3*n trainable
    parameters total (n=4 qubits for MNIST-PCA-8 -> 12 parameters).
    """
    n = num_qubits_for(num_features)
    x = ParameterVector("x", num_features)
    theta = ParameterVector("theta", 3 * n)   # dedicated -> 3*n params, eq. (21)

    qc = QuantumCircuit(n, name="CovariantFeatureMap_dedicated")

    # D(x): eq. (16)
    for k in range(n):
        qc.rx(x[2 * k], k)
        qc.rz(x[2 * k + 1], k)

    # linear entangling layer, eq. (18)
    for k in range(n - 1):
        qc.cz(k, k + 1)

    # Vdedicated(theta): eq. (21) -- independent RXYZ (general U gate) per qubit
    for k in range(n):
        qc.u(theta[3 * k], theta[3 * k + 1], theta[3 * k + 2], k)

    qc.metadata = {"feature_params": list(x), "theta_params": list(theta)}
    return qc


def make_trainable_kernel(num_features: int):
    """TrainableFidelityStatevectorKernel over the CovariantFeatureMap_dedicated circuit."""
    qc = build_covariant_circuit_dedicated(num_features)
    theta_params = qc.metadata["theta_params"]
    kernel = TrainableFidelityStatevectorKernel(feature_map=qc, training_parameters=theta_params)
    return kernel, qc


def make_fixed_kernel(num_features: int, theta_values):
    """Bind theta -> plain (non-trainable) FidelityStatevectorKernel for fast CV/eval."""
    qc = build_covariant_circuit_dedicated(num_features)
    theta_params = qc.metadata["theta_params"]
    bound = qc.assign_parameters(dict(zip(theta_params, theta_values)))
    return FidelityStatevectorKernel(feature_map=bound)


# ------------------------------------------------------------------
# 4. Dataset: MNIST 3-vs-5, PCA-8
# ------------------------------------------------------------------
N_COMPONENTS = 8
N_TRAIN, N_TEST = 250, 250
C_GRID = [0.01, 0.1, 1, 10, 100]
LAMBDA_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]
N_FOLDS = 5


def load_mnist_3_vs_5(path=MNIST_PKL):
    with gzip.open(path, "rb") as f:
        train, val, test = pickle.load(f, encoding="latin1")
    X = np.concatenate([train[0], val[0], test[0]], axis=0)
    y = np.concatenate([train[1], val[1], test[1]], axis=0)
    mask = (y == 3) | (y == 5)
    X, y = X[mask], y[mask]
    y = np.where(y == 3, -1, 1)
    return X, y


def build_pca8_pool(X, y, pca_fit_size=6000, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X), size=min(pca_fit_size, len(X)), replace=False)
    pca = PCA(n_components=N_COMPONENTS, random_state=seed)
    pca.fit(X[idx])
    return pca.transform(X), y, pca


# ------------------------------------------------------------------
# 5. QKT training step (weighted alignment / SVCLoss, eq. 13, 2-SPSA)
# ------------------------------------------------------------------
def train_theta(X_tr, y_tr, qkt_maxiter, seed):
    """
    Optimize the 12 dedicated fiducial-state angles (3 per qubit, eq. 21)
    via weighted kernel-target alignment (SVCLoss = eq. 13) using 2nd-order
    SPSA (paper Section III-C: 400 iterations, automatic calibration, theta
    initialized to all zeros). SPSA's per-iteration cost is dimension-free
    (it only needs O(1) kernel evaluations regardless of parameter count),
    so 12 params costs about the same per iteration as the shared/3-param
    case. Uses lambda=1.0 (no extra bandwidth scaling) for the training
    data, matching the paper's separation of QKT (theta) from the later
    (C, lambda) grid search.
    """
    trainable_kernel, _ = make_trainable_kernel(N_COMPONENTS)
    optimizer = SPSA(maxiter=qkt_maxiter, second_order=True)
    # Paper initializes theta = 0 (identity fiducial state). In practice this
    # is an exactly-flat point for the *dedicated* ansatz's loss landscape along
    # some calibration directions, which trips SPSA's automatic
    # learning-rate calibration (divide-by-zero -> NaN parameters). A tiny
    # symmetry-breaking nudge away from zero fixes this while remaining
    # numerically indistinguishable from the paper's theta=0 start.
    n_qubits = num_qubits_for(N_COMPONENTS)
    n_params = 3 * n_qubits   # dedicated -> 12 params for MNIST-PCA-8 (4 qubits)
    rng = np.random.RandomState(seed)
    initial_point = rng.uniform(-1e-2, 1e-2, size=n_params).tolist()
    qkt = QuantumKernelTrainer(
        quantum_kernel=trainable_kernel,
        loss="svc_loss",
        optimizer=optimizer,
        initial_point=initial_point,
    )
    result = qkt.fit(X_tr, y_tr)
    theta_opt = np.array(list(result.optimal_parameters.values()))

    if not np.all(np.isfinite(theta_opt)):
        # Rare: 2-SPSA's Hessian estimate occasionally diverges. Fall back to
        # plain (1st-order) SPSA, which is much more numerically robust.
        optimizer = SPSA(maxiter=qkt_maxiter, second_order=False)
        qkt = QuantumKernelTrainer(
            quantum_kernel=trainable_kernel, loss="svc_loss",
            optimizer=optimizer, initial_point=initial_point,
        )
        result = qkt.fit(X_tr, y_tr)
        theta_opt = np.array(list(result.optimal_parameters.values()))

    return theta_opt, result.optimal_value


# ------------------------------------------------------------------
# 6. QKE evaluation, CV hyperparameter search (same protocol as plain QKE)
# ------------------------------------------------------------------
def quantum_kernel_matrix(kernel, X_a, X_b, lam):
    return kernel.evaluate(x_vec=lam * X_a, y_vec=lam * X_b)


def cv_select_hyperparams(kernel, X_tr, y_tr, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    folds = list(skf.split(X_tr, y_tr))
    best_acc, best_lam, best_C = -1, LAMBDA_GRID[0], C_GRID[0]
    for lam in LAMBDA_GRID:
        K_full = quantum_kernel_matrix(kernel, X_tr, X_tr, lam)
        for C in C_GRID:
            accs = []
            for tr_idx, val_idx in folds:
                K_tr = K_full[np.ix_(tr_idx, tr_idx)]
                K_val = K_full[np.ix_(val_idx, tr_idx)]
                svc = SVC(kernel="precomputed", C=C)
                svc.fit(K_tr, y_tr[tr_idx])
                accs.append(accuracy_score(y_tr[val_idx], svc.predict(K_val)))
            mean_acc = float(np.mean(accs))
            if mean_acc > best_acc:
                best_acc, best_lam, best_C = mean_acc, lam, C
    return best_lam, best_C


def run_repetition(X_pool, y_pool, seed, qkt_maxiter):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_pool, y_pool, train_size=N_TRAIN, test_size=N_TEST,
        stratify=y_pool, random_state=seed,
    )
    scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    # --- QKT: optimize theta on the training set (eq. 13) ---
    t0 = time.time()
    theta_opt, opt_loss = train_theta(X_tr, y_tr, qkt_maxiter, seed)
    qkt_time = time.time() - t0

    # --- fixed-theta kernel, then usual (C, lambda) CV grid search ---
    kernel = make_fixed_kernel(N_COMPONENTS, theta_opt)
    lam, C = cv_select_hyperparams(kernel, X_tr, y_tr, seed)

    K_train = quantum_kernel_matrix(kernel, X_tr, X_tr, lam)
    K_test = quantum_kernel_matrix(kernel, X_te, X_tr, lam)

    svc = SVC(kernel="precomputed", C=C)
    svc.fit(K_train, y_tr)
    pred_tr = svc.predict(K_train)
    pred_te = svc.predict(K_test)

    return {
        "acc_train": accuracy_score(y_tr, pred_tr),
        "acc_test": accuracy_score(y_te, pred_te),
        "kappa_train": cohen_kappa_score(y_tr, pred_tr),
        "kappa_test": cohen_kappa_score(y_te, pred_te),
        "f1_train": f1_score(y_tr, pred_tr, average="macro"),
        "f1_test": f1_score(y_te, pred_te, average="macro"),
        "lambda": lam, "C": C,
        "theta_opt": theta_opt, "qkt_loss": opt_loss, "qkt_time": qkt_time,
    }


# ------------------------------------------------------------------
# 7. Run everything
# ------------------------------------------------------------------
REPS = 5            # <-- paper uses 30; bump up once you've sanity-checked
QKT_MAXITER = 30    # <-- paper uses 400; each iteration costs ~4-7s at 250 pts
SEED = 0

print("\nLoading MNIST 3-vs-5 and building PCA-8 pool ...")
X, y = load_mnist_3_vs_5()
X_pca, y_pca, pca = build_pca8_pool(X, y, seed=SEED)
print(f"Pool size: {len(X_pca)}  (explained var. ratio sum = "
      f"{pca.explained_variance_ratio_.sum():.3f})")
print(f"QKT_MAXITER={QKT_MAXITER}, REPS={REPS}  "
      f"(paper protocol: QKT_MAXITER=400, REPS=30 -- increase for closer reproduction)\n")

all_metrics = []
t0 = time.time()
for r in range(REPS):
    m = run_repetition(X_pca, y_pca, seed=SEED + r, qkt_maxiter=QKT_MAXITER)
    all_metrics.append(m)
    print(f"  rep {r+1:2d}/{REPS}  acc_test={m['acc_test']:.3f}  "
          f"theta_opt={np.round(m['theta_opt'], 3)}  qkt_loss={m['qkt_loss']:.3f}  "
          f"lambda={m['lambda']}  C={m['C']}  "
          f"(qkt {m['qkt_time']:.1f}s, total {time.time()-t0:.1f}s elapsed)")

keys = ["acc_train", "acc_test", "kappa_train", "kappa_test", "f1_train", "f1_test"]
summary = {k: (np.mean([m[k] for m in all_metrics]),
                np.std([m[k] for m in all_metrics])) for k in keys}

print("\n=== Table 2 format ===")
print(f"{'Dataset':<14}{'Feature map':<20}{'Strategy':<16}"
      f"{'acc_TR':<16}{'acc_TS':<16}{'kappa_TR':<16}{'kappa_TS':<16}"
      f"{'F1_TR':<16}{'F1_TS':<16}")
print(f"{'MNIST-PCA-8':<14}{'CovariantFeatureMap':<20}{'dedicated (12)':<16}"
      f"{summary['acc_train'][0]:.3f}({summary['acc_train'][1]:.3f})   "
      f"{summary['acc_test'][0]:.3f}({summary['acc_test'][1]:.3f})   "
      f"{summary['kappa_train'][0]:.3f}({summary['kappa_train'][1]:.3f})   "
      f"{summary['kappa_test'][0]:.3f}({summary['kappa_test'][1]:.3f})   "
      f"{summary['f1_train'][0]:.3f}({summary['f1_train'][1]:.3f})   "
      f"{summary['f1_test'][0]:.3f}({summary['f1_test'][1]:.3f})")

MNIST file ready: /content/mnist.pkl.gz 17051982 bytes

Loading MNIST 3-vs-5 and building PCA-8 pool ...
Pool size: 13454  (explained var. ratio sum = 0.476)
QKT_MAXITER=30, REPS=5  (paper protocol: QKT_MAXITER=400, REPS=30 -- increase for closer reproduction)

  rep  1/5  acc_test=0.944  theta_opt=[-19.9     4.473  -5.407 -15.745 -11.271  10.463  26.361 -10.56  -19.64
  38.934 -21.348  12.839]  qkt_loss=126.848  lambda=0.5  C=1  (qkt 163.9s, total 170.0s elapsed)
  rep  2/5  acc_test=0.964  theta_opt=[ 23.731   8.985  -6.491  11.281  56.561 -48.42  -25.083 -10.806  23.039
 -38.131  -2.009  -8.094]  qkt_loss=103.242  lambda=0.5  C=10  (qkt 151.2s, total 327.1s elapsed)
  rep  3/5  acc_test=0.940  theta_opt=[  2.96   -4.949  17.86    5.033  -2.226 -22.794  -3.747 -14.974  -2.317
   9.618  -3.223  -5.014]  qkt_loss=109.292  lambda=0.5  C=1  (qkt 150.4s, total 483.9s elapsed)
  rep  4/5  acc_test=0.940  theta_opt=[ -3.662  22.653 -24.449 -14.504 -27.726   2.51    5.294 -24.525  -9.289
   